In [1]:
import json                                 # writes feature_stats, used in the feature-stack cell
import sys                                  # used to edit Python's import search path below
from pathlib import Path                    # file paths as objects, works on any OS

import geopandas as gpd                     # pandas, except every row also carries a shape
import numpy as np                          # arrays and the random number generator
import pandas as pd                         # plain tables (the manifest CSV)
import rasterio                             # read and write GeoTIFFs
from pyparsing import LineStart
from rasterio.features import rasterize     # turn vector shapes into a grid of numbers
from shapely.geometry import box            # build a rectangle from its four edges
from shapely.geometry import LineString
# from notebooks.wellsight.wellsight_morphology_pipeline import pits_in_tile_geodataframe

parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))
from _common import DERIV, path_for, read_layer
RNG_SEED = 42

target_folder = "OTHER_DATA"
SPLIT_FRACS = {"train": 0.70, "val": 0.15, "test": 0.15}
SRC = path_for("derived") / f"{target_folder}" / "derived"
print(SRC)
ANN = path_for("truth") / "annotations_proj.gpkg"
ROAD_BUFFER_M = 1.5
DRAIN_BUFFER_M = 2.0  # channels are a touch wider than the 1.5 m road half-width
ref_path = Path.cwd().parent.parent.parent/"data"/ target_folder / "derived"
ANN = path_for("truth") / "annotations_proj.gpkg"  # your drawings, reprojected to metres
REF = list(ref_path.glob("*dem*.tif"))[0]
OUT = ref_path
GRID_BLOCK_X_DIM, GRID_BLOCK_Y_DIM = 12, 12
SAMPLE_STRIDE_M = 4.0
CHUNK_M = 40.0


C:\Users\colto\Documents\GitHub\lidar_project\data\OTHER_DATA\derived


In [2]:

with rasterio.open(REF) as dem:
    profile = dem.profile.copy()
    transform = dem.transform
    heigh_var, width_var = dem.height, dem.width
    bounds_var = dem.bounds
    crs_var = dem.crs
pit_interior_gdf = read_layer(ANN, "pit_inside")
pit_wall_gdf = read_layer(ANN, "pit_wall")
pad_gdf = read_layer(ANN, "pad")

In [3]:
shapes_wall_lst = [(g, 2) for g in pit_wall_gdf.geometry if g and not g.is_empty]
shapes_floor_lst = [(g, 1) for g in pit_interior_gdf.geometry if g and not g.is_empty]
label = rasterize(shapes_wall_lst + shapes_floor_lst, out_shape=(heigh_var, width_var), transform=transform, fill=0,dtype = "uint8", all_touched=False)
# floor_counter = int((label==1).sum())
# wall_counter = int((label == 2).sum())
# pixel_ground_area = transform.a * (-transform.e)

label_profile = profile.copy()
label_profile.update(dtype = "uint8", count = 1, nodata=255, compress = "deflate", predictor = 2)
label_path = OUT / f"labels_pit_{target_folder}_1m.tif"
with rasterio.open(label_path, mode="w", **label_profile) as dst:
    dst.write(label, 1)


In [4]:
pad_lst = [(g, 2) for g in pad_gdf.geometry if g and not g.is_empty]
pad_mask = rasterize(pad_lst, out_shape=(heigh_var, width_var), transform=transform, fill=0,dtype = "uint8")
pad_path = OUT / f"mask_pad_{target_folder}_1m.tif"
with rasterio.open(pad_path, "w", **label_profile) as dst:
    dst.write(pad_mask, 1)


Spatial Block Grid

In [5]:
minx, miny, maxx, maxy = bounds_var.left, bounds_var.bottom, bounds_var.right, bounds_var.top
block_width_var = (maxx - minx) / GRID_BLOCK_X_DIM
block_height_var = (maxy - miny) / GRID_BLOCK_Y_DIM
blocks_lst = []
for i in range(GRID_BLOCK_X_DIM):
    for j in range(GRID_BLOCK_Y_DIM):
        x0 = minx + i * block_width_var
        y0 = miny + j * block_height_var
        blocks_lst.append({"block_id": j * GRID_BLOCK_X_DIM + i,     # 0..143, one number per block
               "ix": i, "iy": j,               # column and row, handy when debugging
               "geometry": box(x0, y0, x0 + block_width_var, y0 + block_height_var)})  # the block itself
# blocks_gdf = gpd.GeoDataFrame(blocks_lst, crs=pits_in_tile_geodataframe.crs)
blocks_gdf = gpd.GeoDataFrame(blocks_lst, crs=crs_var)



Count Picks per block via centroid

In [6]:
pit_interior_gdf = pit_interior_gdf.copy()
pit_interior_gdf['centroid_x'] = pit_interior_gdf.geometry.centroid.x
pit_interior_gdf['centroid_y'] = pit_interior_gdf.geometry.centroid.y
centers_gdf = gpd.GeoDataFrame(pit_interior_gdf[['pit_inside_id']].copy(), geometry = pit_interior_gdf.geometry.centroid, crs = pit_interior_gdf.crs)
j_holder_gdf = gpd.sjoin(centers_gdf, blocks_gdf[["block_id", "geometry"]], how = "left", predicate="within")
pit_block_gdf = j_holder_gdf.set_index("pit_inside_id")["block_id"].to_dict()
pit_interior_gdf["block_id"]= pit_interior_gdf["pit_inside_id"].map(pit_block_gdf)
print(pit_interior_gdf.columns)
pits_per_block = pit_interior_gdf.groupby("block_id").size().rename("number_pits")
blocks_gdf = blocks_gdf.merge(pits_per_block, on="block_id", how="left")
blocks_gdf["number_pits"] = blocks_gdf["number_pits"].fillna(0).astype(int)


Index(['id', 'pit_inside_id', 'pad_id', 'geometry', 'centroid_x', 'centroid_y',
       'block_id'],
      dtype='str')


In [7]:
pit_blocks_gdf = blocks_gdf[blocks_gdf["number_pits"] > 0][['block_id', 'number_pits']].sample(frac=1, random_state=RNG_SEED).values
print(pit_blocks_gdf)
total_pits_int = int(sum(n for _, n in pit_blocks_gdf))
targets_dict = {s: total_pits_int * f for s, f in SPLIT_FRACS.items()}
running_dict = {"train":0, "val": 0, "test": 0}
split_of_dict = {}
for bid, number_pits in pit_blocks_gdf:
    deficits = {s: targets_dict[s] - running_dict[s] for s in running_dict}
    chosen_data = max(deficits, key=deficits.get)
    split_of_dict[int(bid)] = chosen_data
    running_dict[chosen_data] += int(number_pits)
blocks_gdf['split'] = blocks_gdf["block_id"].map(split_of_dict).fillna("unused")
blocks_gdf.to_file(OUT / f"pit_blocks_{target_folder}.gpkg", driver="GPKG", layer="blocks")
pit_interior_gdf["split"] = pit_interior_gdf["block_id"].map(split_of_dict).fillna("unused")
manifest = pit_interior_gdf[["pit_inside_id", "pad_id", "block_id", "split", "centroid_x", "centroid_y"]].copy()
manifest.columns = ["pit_inside_id", "pad_id", "block_id", "split", "centroid_x", "centroid_y"]
manifest.to_csv(OUT / "pit_dataset_manifest.csv", index=False)
# print(f"Wrote pit_dataset_manifest.csv  ({len(manifest)} pits)")


[[ 44   1]
 [ 48   9]
 [ 87   8]
 [114   2]
 [120   5]
 [ 40   2]
 [126   2]
 [ 78   6]
 [ 64   8]
 [132   3]
 [ 70   4]
 [ 39   5]
 [ 92   4]
 [131   8]
 [ 73   9]
 [  0   5]
 [ 43   5]
 [ 50  10]
 [130   2]
 [  5   1]
 [ 93   3]
 [119   7]
 [  1   1]
 [122   2]
 [ 90   5]
 [ 81   1]
 [ 83   7]
 [  2   5]
 [ 28   7]
 [ 71   6]
 [115   5]
 [ 37   2]
 [ 98   2]
 [ 80   2]
 [108   7]
 [  3   2]
 [ 18   2]
 [ 66   8]
 [ 74   8]
 [111   3]
 [ 60   5]
 [ 41   5]
 [ 54   8]
 [ 75   4]
 [ 27   6]
 [ 49   8]
 [ 95   8]
 [ 15   5]
 [ 53   4]
 [ 84  10]
 [135   6]
 [102   1]
 [118   5]
 [ 62   9]
 [ 85   1]
 [ 57   1]
 [117   4]
 [ 38   4]
 [ 96   1]
 [104   7]
 [ 88   4]
 [ 13   5]
 [ 20   3]
 [ 26   1]
 [ 36   3]
 [ 61   8]
 [ 63   4]
 [140   4]
 [ 72   6]
 [ 55   8]
 [141   6]
 [ 45   1]
 [ 29   1]
 [100   1]
 [ 91   4]
 [ 52   4]
 [103   1]
 [137   2]
 [107   6]
 [127   2]
 [ 21   1]
 [ 99   6]
 [ 77   8]
 [ 76   8]
 [129   7]
 [ 65   7]
 [ 79   6]
 [134   5]
 [ 59  10]
 [ 89   3]
 [ 42   3]

roadprep

In [8]:
channels = [
    ("lrm_25",       f"lrm_25_{target_folder}.tif"),
    ("lrm_5",       f"lrm_5_{target_folder}.tif"),
    ("slope",       f"slope_{target_folder}.tif"),
    ("tpi_05",       f"tpi_05_{target_folder}.tif"),
    ("openness_pos",       f"openness_pos_{target_folder}.tif"),
    ("openness_neg",       f"openness_neg_{target_folder}.tif"),
    ("roughness_5",       f"roughness_5_{target_folder}.tif"),
]
# def main_roads_process():
paths = [(name, SRC/fname) for name, fname in channels]
print(paths)
for _, p in paths:
    if not p.exists():
        raise FileNotransformoundError(p)

# The grid was established from the DEM back in cell 1. Check lrm_25 sits on it
# rather than quietly adopting a second grid from a different file.
with rasterio.open(paths[0][1]) as r0:
    if (r0.height, r0.width) != (heigh_var, width_var) or r0.transform != transform:
        raise ValueError("lrm_25 does not sit on the DEM grid")
height, width = heigh_var, width_var
crs = crs_var

# The 7-band stack needs its own profile. Keep it separate from `profile`, which
# is the DEM's single-band one that the label cells still rely on.
out_path = ref_path / f"features_pit_{target_folder}_1m.tif"
features_profile = profile.copy()
features_profile.update(count=len(paths), dtype="float32", compress="deflate",
               predictor=3, tiled=True, blockxsize=512, blockysize=512,
               BIGTIFF="YES", nodata=np.nan)

# blocks_gdf is already in memory: built in cell 5, given its split column in
# cell 8. Re-reading the gpkg written two cells ago would hand back the same
# frame and overwrite anything added to it since.
training_blocks = blocks_gdf[blocks_gdf["split"] == "train"]
training_mask = rasterize([(g,1) for g in training_blocks.geometry],
                          out_shape=(height, width),transform=transform,
                          fill=0, dtype ="uint8").astype(bool)
stats_dict = {}
with rasterio.open(out_path, "w", **features_profile) as dst:
    for i, (name, src) in enumerate(paths, start=1):
        with rasterio.open(src) as r:
            if (r.height, r.width) != (height, width) or r.transform != transform:
                raise ValueError(f"{name} does not match the DEM grid")
            arr = r.read(1).astype("float32")
            if r.nodata is not None and np.isfinite(r.nodata):
                arr[arr == r.nodata] = np.nan
        vals = arr[training_mask]
        vals = vals[np.isfinite(vals)]
        avg_val, stan_dev = float(np.mean(vals)), float(np.std(vals))
        stats_dict[name] = {'mean': avg_val, 'std': stan_dev, 'p2': float(np.percentile(vals, 2)),
        'p98': float(np.percentile(vals, 98))}
        dst.write(arr, i)
        dst.set_band_description(i, name)
with open(ref_path / f"feature_stats_1m.json", "w") as f:
    json.dump(stats_dict, f, indent=2)


[('lrm_25', WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/OTHER_DATA/derived/lrm_25_OTHER_DATA.tif')), ('lrm_5', WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/OTHER_DATA/derived/lrm_5_OTHER_DATA.tif')), ('slope', WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/OTHER_DATA/derived/slope_OTHER_DATA.tif')), ('tpi_05', WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/OTHER_DATA/derived/tpi_05_OTHER_DATA.tif')), ('openness_pos', WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/OTHER_DATA/derived/openness_pos_OTHER_DATA.tif')), ('openness_neg', WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/OTHER_DATA/derived/openness_neg_OTHER_DATA.tif')), ('roughness_5', WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/OTHER_DATA/derived/roughness_5_OTHER_DATA.tif'))]


road process part 2


In [9]:
roads_gdf = read_layer(ANN, "roads")
layers_gdf = gpd.list_layers(ANN)['name'].tolist()
drainage_gdf = (read_layer(ANN, "drainage") if "drainage" in layers_gdf else gpd.GeoDataFrame(geometry = []))
not_roads_gdf = read_layer(ANN, "not_roads")
label = np.zeros((heigh_var, width_var), dtype="uint8")
if len(drainage_gdf):
    drain_buffer = [g.buffer(DRAIN_BUFFER_M) for g in drainage_gdf.geometry if g is not None and not g.is_empty]
    drain_arr = rasterize([(g, 1) for g in drain_buffer], out_shape=(heigh_var, width_var), transform=transform, fill=0, dtype="uint8")
    label[drain_arr == 1] = 2

road_buffer = [g.buffer(ROAD_BUFFER_M) for g in roads_gdf.geometry if g is not None and not g.is_empty]
road_arr = rasterize([(g,1) for g in road_buffer], out_shape=(heigh_var, width_var), transform=transform, fill=0, dtype="uint8")
label[road_arr == 1] = 1
# label_profile, from cell 2, is already the DEM grid as uint8 / 1 band /
# nodata 255. Rebuilding it from `profile` is what made the blockxsize pops
# necessary -- `profile` had become the tiled 7-band stack.
with rasterio.open(ref_path / f"labels_road_{target_folder}_1m.tif", "w", **label_profile) as dst:
    dst.write(label, 1)



pads and roads

In [10]:
def chunk_line(line: LineString, chunk_m: float):
    if line is None or line.is_empty:
        return []
    if line.length <=  chunk_m:
        return [line]
    n = int(np.ceil(line.length / chunk_m))
    edges = np.linspace(0, line.length, n + 1)
    out=[]
    for t0, t1 in zip(edges[:-1], edges[1:]):
        inner = max(2, int(np.ceil((t1-t0) / 5)) + 1)
        ts = np.linspace(t0, t1, inner)
        seg = LineString([(p.x, p.y) for p in (line.interpolate(t) for t in ts)])
        if not seg.is_empty and seg.length > 0:
            out.append(seg)
    return out
def assign_split(geom_or_pt, blocks):
    if geom_or_pt is None or getattr(geom_or_pt, "is_empty", True):
        return None, "unused"
    if hasattr(geom_or_pt, "centroid"):
        pt = geom_or_pt.centroid if geom_or_pt.geom_type != "Point" else geom_or_pt
    else:
        pt = geom_or_pt
    hit = blocks[blocks.geometry.contains(pt)]
    if len(hit):
        return int(hit.iloc[0].block_id), str(hit.iloc[0]. split)
    return None, "unused"
def sample_along_line(line: LineString, stride_m: float):
    pts = []
    line_length = line.length
    if line_length < 1:
        pts.append(line.interpolate(0.5, normalized=True))
        return pts
    n = max(1, int(line_length // stride_m))
    for i in range(n+1):
        d = min(i*stride_m, line_length)
        pts.append(line.interpolate(d))
    return pts

Main function

In [11]:
# # The DEM was opened in cell 1 and pad_gdf read there too -- nothing below needs
# # either of them again. transform is just the same transform under a shorter name.
# transform = transform
#
# # The only layer this section actually adds. not_roads are lines a human looked
# # at and ruled out, so they are hard negatives rather than missing data.
#
#
# # blocks_gdf is already in memory, split column and all, from cells 5 and 8.
# # (The line here read the gpkg back -- and passed the GeoDataFrame itself as the
# # path, so it could not have run.)
# # roads = read_layer(ANN, "roads")
# # drainage_gdf = (read_layer(ANN, "drainage") if "drainage" in gpd.list_layers(ANN)["name"].tolist() else gpd.GeoDataFrame(geometry = [], crs = roads.crs))
#
#




In [12]:
# pad label raster
pad_arr = rasterize( [(g, 1) for g in pad_gdf.geometry if g and not g.is_empty], out_shape=(heigh_var, width_var), transform=transform, fill=0, dtype="uint8")
p1 = profile.copy()
p1.update(dtype = "uint8", count=1, nodata=255, compress = "deflate", predictor=2)
with rasterio.open(ref_path / f"labels_pad_{target_folder}_1m.tif", "w", **p1) as dst:
    dst.write(pad_arr,1)

In [13]:
# road label raster
buffered_lst = [g.buffer(ROAD_BUFFER_M) for g in roads_gdf.geometry if g and not g.is_empty]
road_arr = rasterize([(g,1) for g in buffered_lst], out_shape=(heigh_var, width_var), transform=transform, fill=0, dtype="uint8")
with rasterio.open(ref_path / f"labels_road_binary_{target_folder}_1m.tif", "w", **p1) as dst:
    dst.write(road_arr, 1)

In [14]:
# pad manifest
pad_rows_lst = []
for _, row in pad_gdf.iterrows():
    if row.geometry is None or row.geometry.is_empty:
        continue
    bid, split = assign_split(row.geometry, blocks_gdf)
    centroid = row.geometry.centroid
    pad_rows_lst.append({'pad_id': int(row.pad_id), "block_id": bid, "split": split, "centroid_x": centroid.x, "centroid_y": centroid.y, "area_m2": float(row.geometry.area)})
pad_manifest = pd.DataFrame(pad_rows_lst)
pad_manifest.to_csv(ref_path / "pad_dataset_manifest.csv", index=False)

In [15]:
# road chunk manifest
chunk_rows_lst = []
chunk_geoms_lst = []
for src, kind, do_chunk in [(roads_gdf, "road", True),
                           (not_roads_gdf, "not_road", True),
                           (drainage_gdf, "drainage", False)]:
    for i, row in src.reset_index(drop=True).iterrows():
        row_geom = row.geometry
        if row_geom is None or row_geom.is_empty:
            continue
        segments = chunk_line(row_geom, CHUNK_M) if do_chunk else [row_geom]
        for j, seg in enumerate(segments):
            mid_point = seg.interpolate(0.5, normalized=True)
            bid, split = assign_split(mid_point, blocks_gdf)
            lid = f"{kind}_{i}_{j}"
            chunk_rows_lst.append({"line_id": lid,
                                   "kind": kind,
                                   "parent_id": i,
                                   'seg_idx': j,
                                   'block_id': bid,
                                   "split": split,
                                   "mid_x": mid_point.x,
                                   "mid_y": mid_point.y,
                                   "length_m": float(seg.length)})
            chunk_geoms_lst.append({"line_id": lid,"kind": kind,"split": split, "geometry": seg})
road_man_df = pd.DataFrame(chunk_rows_lst)
road_man_df.to_csv(ref_path / "road_dataset_manifest.csv", index=False)
# An empty list would build a frame with no geometry column, and to_file
# raises on that -- which is exactly what happens on a tile holding no roads.
if chunk_geoms_lst:
    gchunks_gdf = gpd.GeoDataFrame(chunk_geoms_lst, crs = roads_gdf.crs)
    gchunks_gdf.to_file(ref_path/ f"road_chunks_{target_folder}.gpkg", layer="chunks", driver = "gpkg")
    print(f"{ref_path / f'road_chunks_{target_folder}.gpkg'}  ({len(gchunks_gdf)} chunks)")
else:
    print("no road chunks -- nothing written")


C:\Users\colto\Documents\GitHub\lidar_project\data\OTHER_DATA\derived\road_chunks_OTHER_DATA.gpkg  (17591 chunks)


In [16]:
#road classifier sample points
pt_rows_lst = []
for src, kind, label in [(roads_gdf, "road", 1), (not_roads_gdf, "not_road", 0)]:
    for i, row in src.reset_index(drop=True).iterrows():
        line_id = f"{kind}_{i}"
        if row.geometry is None or row.geometry.is_empty:
            continue
        for pt in sample_along_line(row.geometry, SAMPLE_STRIDE_M):
            bid, split = assign_split(pt, blocks_gdf)
            pt_rows_lst.append({"line_id": line_id,"kind": kind,"label": label, "block_id": bid, "split": split, "x":pt.x, "y":pt.y})
pts_df = pd.DataFrame(pt_rows_lst)
pts_df.to_csv(ref_path / "road_classifier_samples.csv", index=False)
by_split_df = pts_df.groupby(['label', "split"]).size().unstack(fill_value=0)


In [17]:
pad_gdf = pad_gdf[~pad_gdf.geometry.isna() & ~pad_gdf.geometry.is_empty].copy()
pad_gdf['ctr_x'] = pad_gdf.geometry.centroid.x
pad_gdf['ctr_y'] = pad_gdf.geometry.centroid.y

pad_centroid_gdf = gpd.GeoDataFrame(
    pad_gdf[["pad_id", 'ctr_x', 'ctr_y']],
    geometry=gpd.points_from_xy(pad_gdf.ctr_x, pad_gdf.ctr_y), crs = pad_gdf.crs
)
joined_gdf = gpd.sjoin(pad_centroid_gdf, blocks_gdf, how='left', predicate='within')
n_per_block = joined_gdf.groupby("block_id").size()
blocks_gdf["n_pads"] = blocks_gdf["block_id"].map(n_per_block).fillna(0).astype(int)



In [ ]:
#greedy fill by pad count
total = int(blocks_gdf["n_pads"].sum())
print(total)
pad_blocks = blocks_gdf[blocks_gdf.n_pads > 0][["block_id", "n_pads"]].sample(frac=1, random_state=RNG_SEED).values
print(pad_blocks)
targets_dict  = {s: total * f for s, f in SPLIT_FRACS.items()}
running_dict  = {s: 0 for s in SPLIT_FRACS}
split_of_dict = {}
for bid, n in pad_blocks:
    chosen = max(SPLIT_FRACS, key=lambda s: targets_dict[s] - running_dict[s])
    split_of_dict[int(bid)] = chosen
    running_dict[chosen] += int(n)
blocks_gdf["split"] = blocks_gdf["block_id"].map(split_of_dict).fillna("unused")
print(running_dict)

# .gpkg, not .csv -- the driver here is GPKG, and everything downstream globs
# for pad_blocks_*.gpkg. The pit version two cells up is the naming to match.
blocks_gdf.to_file(ref_path / f"pad_blocks_{target_folder}.gpkg", layer = "blocks", driver="GPKG")


In [ ]:
# pad manifest, rebuilt on the pad split
# The one written earlier carries the PIT split, because blocks_gdf still held it
# then. The cell above just replaced that column, so the pads have to be re-read
# against it or the file on disk and the blocks file disagree.
pad_rows_lst = []
for _, row in pad_gdf.iterrows():
    if row.geometry is None or row.geometry.is_empty:
        continue
    bid, split = assign_split(row.geometry, blocks_gdf)
    centroid = row.geometry.centroid
    pad_rows_lst.append({'pad_id': int(row.pad_id), "block_id": bid, "split": split, "centroid_x": centroid.x, "centroid_y": centroid.y, "area_m2": float(row.geometry.area)})
pad_manifest = pd.DataFrame(pad_rows_lst)
pad_manifest.to_csv(ref_path / "pad_dataset_manifest.csv", index=False)
print(pad_manifest["split"].value_counts().to_dict())


In [ ]:
# what phase 3 produced -- run this last, it is the handover to training
# Anything missing here is an input the U-Net trainer will not find.
expected_lst = [
    f"features_pit_{target_folder}_1m.tif",
    "feature_stats_1m.json",
    f"labels_pit_{target_folder}_1m.tif",
    f"labels_pad_{target_folder}_1m.tif",
    f"labels_road_{target_folder}_1m.tif",
    f"pit_blocks_{target_folder}.gpkg",
    f"pad_blocks_{target_folder}.gpkg",
    f"road_chunks_{target_folder}.gpkg",
    "pit_dataset_manifest.csv",
    "pad_dataset_manifest.csv",
    "road_dataset_manifest.csv",
    "road_classifier_samples.csv",
]
for fname in expected_lst:
    p = ref_path / fname
    print(f"{'ok ' if p.exists() else 'MISSING'}  {p.stat().st_size/1e6:8.2f} MB  {fname}" if p.exists() else f"MISSING            {fname}")

# The two splits are deliberately different: pits are balanced on pit count,
# pads on pad count. The road manifest carries whichever split was on blocks_gdf
# when its cell ran, which is the pit one.
for name in ["pit_dataset_manifest.csv", "pad_dataset_manifest.csv", "road_dataset_manifest.csv"]:
    p = ref_path / name
    if p.exists():
        man_df = pd.read_csv(p)
        print(f"{name:32s} {len(man_df):6d} rows  {man_df['split'].value_counts().to_dict()}")
